# vibiomir-embpage-b-s0: BAAI/bge-m3 pages (max 8192 tokens), input parts 0/2 of thanhdatdo/vibiomir-pages-b

Generated by share/kaggle/mk_embed.py. Text: title + chunk, NFC + quote/whitespace cleanup + one tone placement (textprep.normalize + bm25_normalize); queries the same.

In [ ]:
WORKER = '\nimport torch, torch.nn.functional as F                        # torch before pandas / pyarrow\nimport sys, time, glob, re, unicodedata as ud, numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq\ngpu, ngpu, inp, out, deadline = int(sys.argv[1]), int(sys.argv[2]), sys.argv[3], sys.argv[4], float(sys.argv[5])\nKIND, MODEL, MAXLEN, SHARD, NSHARD = "pages", "BAAI/bge-m3", 8192, 0, 2\nREPL = {"\\n": " ", "‘": "\'", "”": \'"\', "“": \'"\', "′": "\'", "``": "", "\'\'": "", "|": ""}\nOLD_TONE = re.compile(r"(?<![qQ])([oOuU])([aAeEyY])([̣̀́̃̉])(?=\\W|$)")\ndef norm(t):                                                    # textprep.normalize + textprep.bm25_normalize\n    t = ud.normalize("NFC", t or "")\n    for a, b in REPL.items(): t = t.replace(a, b)\n    t = re.sub(r"\\s+", " ", t).strip()\n    t = OLD_TONE.sub(lambda m: m.group(1) + m.group(3) + m.group(2), ud.normalize("NFD", t))\n    return ud.normalize("NFC", t)\ndef concat(title, ctx):\n    title = (title or "").strip()\n    if not title: return ctx or ""\n    return title + (" " if title[-1] in ".!?;:,。！？；：，、" else ". ") + (ctx or "")\nfiles = sorted(glob.glob(f"{inp}/**/{\'chunks\' if KIND == \'chunks\' else \'pages\'}-*.parquet", recursive=True))\nfiles = files[SHARD::NSHARD]                                     # this notebook\'s share of the input parts\nkeys = ["doc_id", "chunk_idx"] if KIND == "chunks" else ["doc_id"]\ncols = keys + (["title", "context", "page_title"] if KIND == "chunks" else ["text"])\ntb = pa.concat_tables([pq.read_table(f, columns=cols) for f in files])\nd = tb.take(pa.array(np.arange(gpu, tb.num_rows, ngpu))).to_pandas(); del tb      # this GPU\'s rows only\nif KIND == "chunks":\n    title = d.title.where(d.title.fillna("").str.strip() != "", d.page_title).fillna("")\n    texts = [norm(concat(a, b)) for a, b in zip(title, d.context)]\nelse:\n    texts = [norm(t) for t in d.text]\nfrom transformers import AutoTokenizer, AutoModel\ntok = AutoTokenizer.from_pretrained(MODEL)\nif MODEL.startswith("intfloat/"): texts = ["passage: " + t for t in texts]\nd = d[keys].reset_index(drop=True)                                # the text columns are no longer needed\n# lengths only (token id lists for ~450k texts as Python ints took ~7 GB per worker: both were OOM-killed, Oct 23);\n# each batch is tokenised again when it is encoded\nL = np.zeros(len(texts), np.int32)\nfor s in range(0, len(texts), 20000):\n    L[s:s + 20000] = [len(x) for x in tok(texts[s:s + 20000], truncation=True, max_length=MAXLEN)["input_ids"]]\norder = np.argsort(-L)                                            # longest first: OOM shows up at once\nm = AutoModel.from_pretrained(MODEL, torch_dtype=torch.float16, attn_implementation="sdpa").cuda().eval()\nSP = None\nif MODEL == "BAAI/bge-m3":\n    from huggingface_hub import hf_hub_download\n    SP = torch.nn.Linear(1024, 1); SP.load_state_dict(torch.load(hf_hub_download(MODEL, "sparse_linear.pt"), map_location="cpu"))\n    SP = SP.half().cuda()\nspecial = {tok.cls_token_id, tok.eos_token_id, tok.pad_token_id, tok.unk_token_id}\n@torch.no_grad()\ndef encode(batch):\n    n = max(len(x) for x in batch)\n    ii = torch.full((len(batch), n), tok.pad_token_id); am = torch.zeros((len(batch), n), dtype=torch.long)\n    for k, x in enumerate(batch): ii[k, :len(x)] = torch.tensor(x); am[k, :len(x)] = 1\n    ii, am = ii.cuda(), am.cuda()\n    H = m(input_ids=ii, attention_mask=am).last_hidden_state\n    if SP is None:                                               # e5: mean pooling\n        v = (H * am[..., None]).sum(1) / am.sum(1, keepdim=True)\n        return F.normalize(v.float(), dim=-1).half().cpu().numpy(), None\n    dense = F.normalize(H[:, 0].float(), dim=-1).half().cpu().numpy()\n    w = torch.relu(SP(H)).squeeze(-1).float() * am\n    w, ii = w.cpu().numpy(), ii.cpu().numpy()\n    sp = []\n    for k in range(len(batch)):\n        acc = {}\n        for t, x in zip(ii[k, :len(batch[k])], w[k, :len(batch[k])]):\n            if t not in special and x > 0 and x > acc.get(t, 0.0): acc[int(t)] = float(x)\n        sp.append((np.fromiter(acc.keys(), np.int32, len(acc)), np.fromiter(acc.values(), np.float32, len(acc))))\n    return dense, sp\npart, buf_d, buf_s, buf_i, t0, done = 0, [], [], [], time.time(), 0\nTOK = 40000 if MAXLEN <= 1024 else 24000                         # tokens per batch (T4 16 GB, fp16, sdpa)\ndef flush():\n    global part, buf_d, buf_s, buf_i\n    if not buf_i: return\n    sel = d.iloc[np.concatenate(buf_i)][keys].reset_index(drop=True)\n    np.save(f"{out}-{gpu}-{part:03d}-dense.npy", np.vstack(buf_d))\n    sel.to_parquet(f"{out}-{gpu}-{part:03d}-ids.parquet", index=False)\n    if buf_s:\n        s = [x for b in buf_s for x in b]\n        t = pa.table({**{k: sel[k].to_numpy() for k in keys}, "tok": pa.array([x[0] for x in s], pa.list_(pa.int32())),\n                      "w": pa.array([x[1] for x in s], pa.list_(pa.float32()))})\n        pq.write_table(t, f"{out}-{gpu}-{part:03d}-sparse.parquet", compression="zstd")\n    part, buf_d, buf_s, buf_i = part + 1, [], [], []\ni = 0\nwhile i < len(order) and time.time() < deadline:\n    n = max(1, min(512, TOK // max(int(L[order[i]]), 16)))\n    idx = order[i:i + n]\n    dv, sv = encode(tok([texts[j] for j in idx], truncation=True, max_length=MAXLEN)["input_ids"])\n    buf_d.append(dv); buf_i.append(idx)\n    if sv is not None: buf_s.append(sv)\n    i += n; done += len(idx)\n    if sum(len(x) for x in buf_i) >= 100_000: flush()\n    if (i // n) % 200 == 0: print(f"gpu{gpu}: {done:,}/{len(order):,} {time.time()-t0:.0f}s len {L[order[min(i, len(order)-1)]]}", flush=True)\nflush()\nprint(f"gpu{gpu}: done {done:,}/{len(order):,} in {time.time()-t0:.0f}s", flush=True)\n'
open('/kaggle/working/w.py', 'w').write(WORKER)

In [ ]:
import os, subprocess, sys, time, torch
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.51.0"], check=False)
from huggingface_hub import snapshot_download; snapshot_download("BAAI/bge-m3")
deadline = time.time() + 11.3 * 3600
n = torch.cuda.device_count()
ps = [subprocess.Popen([sys.executable, "/kaggle/working/w.py", str(g), str(n), "/kaggle/input", "/kaggle/working/emb", str(deadline)],
                       env={**os.environ, "CUDA_VISIBLE_DEVICES": str(g)}) for g in range(n)]
print([p.wait() for p in ps])
os.remove("/kaggle/working/w.py")

In [ ]:

import torch, torch.nn.functional as F
import glob, re, unicodedata as ud, numpy as np, pandas as pd, pyarrow as pa, pyarrow.parquet as pq
from transformers import AutoTokenizer, AutoModel
REPL = {"\n": " ", "‘": "'", "”": '"', "“": '"', "′": "'", "``": "", "''": "", "|": ""}
OLD_TONE = re.compile(r"(?<![qQ])([oOuU])([aAeEyY])([̣̀́̃̉])(?=\W|$)")
def norm(t):
    t = ud.normalize("NFC", t or "")
    for a, b in REPL.items(): t = t.replace(a, b)
    t = re.sub(r"\s+", " ", t).strip()
    t = OLD_TONE.sub(lambda m: m.group(1) + m.group(3) + m.group(2), ud.normalize("NFD", t))
    return ud.normalize("NFC", t)
MODEL = "BAAI/bge-m3"
q = pd.read_parquet(sorted(glob.glob("/kaggle/input/**/queries.parquet", recursive=True))[0])
q["text"] = [norm(t) for t in q["query"]]
tok = AutoTokenizer.from_pretrained(MODEL)
m = AutoModel.from_pretrained(MODEL, torch_dtype=torch.float16).cuda().eval()
texts = [("query: " + t) if MODEL.startswith("intfloat/") else t for t in q.text]
D, S = [], []
SP = None
if MODEL == "BAAI/bge-m3":
    from huggingface_hub import hf_hub_download
    SP = torch.nn.Linear(1024, 1); SP.load_state_dict(torch.load(hf_hub_download(MODEL, "sparse_linear.pt"), map_location="cpu")); SP = SP.half().cuda()
special = {tok.cls_token_id, tok.eos_token_id, tok.pad_token_id, tok.unk_token_id}
with torch.no_grad():
    for s in range(0, len(texts), 32):
        b = tok(texts[s:s+32], padding=True, truncation=True, max_length=512, return_tensors="pt").to("cuda")
        H = m(**b).last_hidden_state
        if SP is None:
            v = (H * b["attention_mask"][..., None]).sum(1) / b["attention_mask"].sum(1, keepdim=True)
            D.append(F.normalize(v.float(), dim=-1).half().cpu().numpy()); continue
        D.append(F.normalize(H[:, 0].float(), dim=-1).half().cpu().numpy())
        w = (torch.relu(SP(H)).squeeze(-1).float() * b["attention_mask"]).cpu().numpy(); ii = b["input_ids"].cpu().numpy()
        for k in range(len(ii)):
            acc = {}
            for t, x in zip(ii[k], w[k]):
                if t not in special and x > 0 and x > acc.get(t, 0.0): acc[int(t)] = float(x)
            S.append((list(acc.keys()), list(acc.values())))
np.save("/kaggle/working/queries_dense.npy", np.vstack(D))
q[["id", "query", "text"]].to_parquet("/kaggle/working/queries.parquet", index=False)
if S:
    pq.write_table(pa.table({"id": q["id"].to_numpy(), "tok": pa.array([x[0] for x in S], pa.list_(pa.int32())),
                             "w": pa.array([x[1] for x in S], pa.list_(pa.float32()))}), "/kaggle/working/queries_sparse.parquet")
print("queries", len(q))
